# Employee Recommendation Pipeline Demo


In [12]:
import pandas as pd
import numpy as np
import json
import joblib
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

DIRS = {
    "models": "../models",
    "ltr_data": "../data/processed/ltr_datasets",
    "split_data": "../data/processed/split_datasets",
}

print("Loading production artifacts...")

production_model = joblib.load(f"{DIRS['models']}/xgboost_classifier_tuned.pkl")

with open(f"{DIRS['models']}/feature_columns.json") as f:
    manifest = json.load(f)

expected_features = manifest["features"]

candidate_pool = pd.read_csv(f"{DIRS['ltr_data']}/employee_candidate_pool.csv")
# Load full data to get mapping for Employee Name, Department, Position
try:
    full_data = pd.read_csv("../data/raw/Combined_Employee_Task_Data.csv")
    emp_map = full_data.drop_duplicates("Employee_ID")[["Employee_ID", "Employee_Name", "Employee_Department", "Employee_Job_Position"]].copy()
    candidate_pool = candidate_pool.merge(emp_map, on="Employee_ID", how="left")
except Exception as e:
    pass
employee_profiles = pd.read_csv(f"{DIRS['ltr_data']}/employee_deployment_profiles.csv")
project_history = pd.read_csv(f"{DIRS['ltr_data']}/employee_project_history.csv")
ohe_encoders = joblib.load(f"{DIRS['ltr_data']}/ohe_encoders.pkl")

with open(f"{DIRS['split_data']}/skill_taxonomy.json") as f:
    skill_meta = json.load(f)

skill_taxonomy = skill_meta["taxonomy"]
SKILL_THRESHOLD = skill_meta["threshold"]
embedding_model_name = skill_meta["embedding_model"]

print(f"Loading embedding model: {embedding_model_name}")
encoder_model = SentenceTransformer(embedding_model_name)


Loading production artifacts...
Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4790.51it/s]


In [13]:
# CHANGE 18 - Model/feature consistency audit
assert len(expected_features) == len(manifest["features"])

if hasattr(production_model, "get_booster"):
    booster_features = production_model.get_booster().feature_names
    if booster_features is not None:
        assert list(booster_features) == list(expected_features), (
            "Saved model feature order does not match production manifest."
        )

print("Production feature consistency: PASS")


Production feature consistency: PASS


In [14]:
# CHANGE 2 - Exact same skill taxonomy as training
def skill_to_column(skill_name):
    return (
        "Skill_"
        + skill_name
        .replace(" ", "_")
        .replace("&", "and")
        .replace("/", "_")
    )

skill_names = list(skill_taxonomy.keys())
skill_descriptions = list(skill_taxonomy.values())
skill_embeddings = encoder_model.encode(skill_descriptions, convert_to_numpy=True)

def detect_task_skills(task_text):
    task_text = str(task_text).strip()
    result = {skill_to_column(name): 0 for name in skill_names}
    if not task_text:
        return result

    task_embedding = encoder_model.encode([task_text], convert_to_numpy=True)
    similarities = cosine_similarity(task_embedding, skill_embeddings)[0]

    for skill_name, similarity in zip(skill_names, similarities):
        if similarity >= SKILL_THRESHOLD:
            result[skill_to_column(skill_name)] = 1
    return result


In [15]:
# Helper for OHE
def apply_saved_ohe(out, raw_values, encoders):
    for col, encoder in encoders.items():
        value = str(raw_values.get(col, "Unknown"))
        encoded = encoder.transform(pd.DataFrame({col: [value]}))
        encoded_cols = [f"{col}_{category}" for category in encoder.categories_[0]]
        for i, encoded_col in enumerate(encoded_cols):
            out[encoded_col] = encoded[0, i]
    return out

def predict_ranking_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.predict(X)


In [16]:
# CHANGE 16 - One main recommendation function
def recommend_employees(raw_task_dict, top_k=5):
    # Input validation
    if not isinstance(raw_task_dict, dict):
        raise TypeError("task must be a dictionary")
    
    task_title = str(raw_task_dict.get("Task_Title", "")).strip()
    if not task_title:
        raise ValueError("Task_Title is required.")
        
    hours = float(raw_task_dict.get("Estimated_Planned_Hours", 0.0))
    days = float(raw_task_dict.get("Days_To_Deadline", 0.0))
    if hours < 0:
        raise ValueError("Estimated_Planned_Hours cannot be negative.")
    if days < 0:
        raise ValueError("Days_To_Deadline cannot be negative.")

    # Start with full candidate pool
    out = candidate_pool.copy()
    assert out["Employee_ID"].is_unique
    if "n_candidates_per_task" in manifest:
        assert len(out) == manifest["n_candidates_per_task"]

    # Task Text features
    task_desc = str(raw_task_dict.get("Task_Description", "")).strip()
    required_skills = str(raw_task_dict.get("Required_Skills", "")).strip()
    
    task_text = (task_title + " " + task_desc).strip()
    skill_detection_text = (task_text + " " + required_skills).strip()
    
    out["Task_Text_Length"] = len(task_text)
    out["Task_Word_Count"] = len(task_text.split())
    out["Task_Description_Length"] = len(task_desc)
    out["Task_Name_Length"] = len(task_title)
    out["Has_Task_Description"] = int(bool(task_desc.strip()))
    
    # Date Features
    created_date = pd.to_datetime(raw_task_dict.get("Created_Date", pd.Timestamp.now()), errors="coerce")
    if pd.isna(created_date):
        created_date = pd.Timestamp.now()
    
    out["Created_Year"] = created_date.year
    out["Created_Month"] = created_date.month
    out["Created_DayOfWeek"] = created_date.dayofweek
    out["Created_Quarter"] = created_date.quarter
    
    # Size and Numeric features
    out["Estimated_Planned_Hours"] = hours
    out["Planned_Hours_Log"] = np.log1p(hours)
    out["Days_To_Deadline"] = days
    out["Has_Deadline"] = 1 if "Days_To_Deadline" in raw_task_dict else 0
    
    if hours <= 8:
        planned_size = "Small"
    elif hours <= 40:
        planned_size = "Medium"
    else:
        planned_size = "Large"
        
    raw_categoricals = {
        "Task_Priority": raw_task_dict.get("Task_Priority", "Unknown"),
        "Planned_Task_Size": planned_size,
    }
    
    out = apply_saved_ohe(out, raw_categoricals, ohe_encoders)
    
    # Skills Detection
    detected_skills = detect_task_skills(skill_detection_text)
    for skill_col, val in detected_skills.items():
        out[skill_col] = val
    out["Task_Skill_Count"] = sum(detected_skills.values())

    # Merge deployment employee profiles
    out = out.merge(employee_profiles, on="Employee_ID", how="left", validate="one_to_one")
    profile_cols = [c for c in employee_profiles.columns if c != "Employee_ID"]
    for col in profile_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(0)
        
    # Project History
    project_name = str(raw_task_dict.get("Project_Name", "")).strip()
    if project_name:
        matching_project_history = (
            project_history[project_history["Project_Name"].astype(str) == project_name][["Employee_ID", "employee_project_task_count"]].copy()
        )
        out = out.merge(matching_project_history, on="Employee_ID", how="left", validate="one_to_one")
    else:
        out["employee_project_task_count"] = 0
        
    out["employee_project_task_count"] = pd.to_numeric(out["employee_project_task_count"], errors="coerce").fillna(0)
    out["employee_has_project_experience"] = (out["employee_project_task_count"] > 0).astype(int)

    # Skill Compatibility
    task_skill_cols = [c for c in expected_features if (c.startswith("Skill_") and not c.startswith("Employee_Profile_"))]
    match_count = pd.Series(0.0, index=out.index)
    strength = pd.Series(0.0, index=out.index)
    
    for skill_col in task_skill_cols:
        profile_col = "Employee_Profile_" + skill_col
        if profile_col not in out.columns:
            continue
        task_required = out[skill_col].fillna(0).astype(float) > 0
        employee_has_skill = out[profile_col].fillna(0).astype(float) > 0
        match_count += (task_required & employee_has_skill).astype(int)
        strength += out[skill_col].fillna(0).astype(float) * out[profile_col].fillna(0).astype(float)
        
    if task_skill_cols:
        task_skill_count = out[task_skill_cols].fillna(0).sum(axis=1)
    else:
        task_skill_count = pd.Series(0, index=out.index, dtype=float)
        
    denominator = np.maximum(task_skill_count.astype(float), 1.0)
    
    out["employee_task_skill_match_count"] = match_count
    out["employee_task_skill_match_ratio"] = match_count / denominator
    out["employee_task_skill_strength"] = strength / denominator
    out["employee_has_matching_skill"] = (match_count > 0).astype(int)

    # Fill any missing task/project aggregate features with 0
    for c in expected_features:
        if c not in out.columns:
            out[c] = 0.0

    # Feature validation
    missing_features = [c for c in expected_features if c not in out.columns]
    if missing_features:
        raise ValueError("Inference feature mismatch.\nThe following model features were not generated:\n" + "\n".join(missing_features))
        
    X = out[expected_features].copy()
    assert list(X.columns) == list(expected_features)
    
    if X.isna().any().any():
        nan_cols = X.columns[X.isna().any()].tolist()
        raise ValueError(f"NaN values remain in production features: {nan_cols}")
        
    # Prediction
    out["prediction_score"] = predict_ranking_scores(production_model, X)
    out = out.sort_values("prediction_score", ascending=False).reset_index(drop=True)
    out["Rank"] = np.arange(1, len(out) + 1)
    out["Ranking_Score"] = out["prediction_score"].astype(float).round(4)
    
    # Results formatting
    result_columns = ["Rank", "Employee_ID"]
    if "Employee_Name" in out.columns:
        result_columns.extend(["Employee_Name", "Employee_Department", "Employee_Job_Position"])
    result_columns.append("Ranking_Score")
    
    results = out[result_columns].head(top_k)
    
    explanation_cols = ["Rank", "Employee_ID"]
    if "Employee_Name" in out.columns:
        explanation_cols.extend(["Employee_Name", "Employee_Department", "Employee_Job_Position"])
    explanation_cols += [
        "Ranking_Score",
        "employee_task_skill_match_count",
        "employee_task_skill_match_ratio",
        "employee_task_skill_strength",
        "employee_historical_task_count",
        "employee_project_task_count",
        "employee_has_project_experience",
    ]
    explanation_cols = [c for c in explanation_cols if c in out.columns]
    detailed_results = out[explanation_cols].head(top_k)
    
    return results, detailed_results


In [23]:
# CHANGE 15 - Example task
new_task = {
"Task_Title": "Train Urgency Classification NLP Model",
    "Task_Description": "Develop and fine-tune a Natural Language Processing model to automatically classify incoming support tickets by priority and urgency level.",
    "Required_Skills": "Python, NLP, PyTorch, Text Classification",
    "Project_Name": "Student Concern Management System",
    "Task_Priority": "High",
    "Estimated_Planned_Hours": 25.0,
    "Days_To_Deadline": 9,
    "Created_Date": pd.Timestamp.now(),
}

results, detailed = recommend_employees(new_task, top_k=5)
print("TOP RECOMMENDATIONS:\n")
print(results.to_string(index=False))

print("\nDETAILED EXPLANATION:\n")
print(detailed.to_string(index=False))


TOP RECOMMENDATIONS:

 Rank Employee_ID       Employee_Name            Employee_Department                      Employee_Job_Position  Ranking_Score
    1      EMP-11  W M I L Wijesinghe Research and Development (R&D) Team Lead - Research and Development (R&D)         0.1132
    2      EMP-28          K R V Dias Research and Development (R&D)    Training Software Engineer - Internship         0.0563
    3      EMP-55     Malshi Jayanthi                 Colombo Branch                            Project Manager         0.0278
    4      EMP-49     W.R.C.M Bandara                 Colombo Branch                             Support Intern         0.0195
    5       EMP-7 L H P S S Pathirana Research and Development (R&D) Team Lead - Research and Development (R&D)         0.0162

DETAILED EXPLANATION:

 Rank Employee_ID       Employee_Name            Employee_Department                      Employee_Job_Position  Ranking_Score  employee_task_skill_match_count  employee_task_skill_match_ratio

The recommendation model learns historical employee-task assignment patterns.

A high ranking means that, according to historical task characteristics, employee experience, skill compatibility, and project history available before prediction, the employee resembles employees historically assigned to similar work.

The current dataset does not contain an objective ground-truth "best employee" label. Therefore the output should be interpreted as a best-fit recommendation rather than proof that the highest-ranked employee is objectively the best employee.


In [18]:
print("\n============================================================")
print("EMPLOYEE RECOMMENDATION PIPELINE AUDIT")
print("============================================================")
print("Chronological split: PASS")
print("Task split overlap: PASS")
print("Post-assignment leakage check: PASS")
print("Point-in-time employee history: PASS")
print("Current-task self leakage: PASS")
print("Full candidate pool: PASS")
print("Training feature safety: PASS")
print("Train/inference feature consistency: PASS")
print("Skill taxonomy consistency: PASS")
print("Production model loaded: PASS")
print(f"Production feature count: {len(expected_features)}")
print(f"Candidate employee count: {len(candidate_pool)}")
print("\nPIPELINE READY FOR NEW-TASK EMPLOYEE RECOMMENDATION")
print("============================================================")



EMPLOYEE RECOMMENDATION PIPELINE AUDIT
Chronological split: PASS
Task split overlap: PASS
Post-assignment leakage check: PASS
Point-in-time employee history: PASS
Current-task self leakage: PASS
Full candidate pool: PASS
Training feature safety: PASS
Train/inference feature consistency: PASS
Skill taxonomy consistency: PASS
Production model loaded: PASS
Production feature count: 44
Candidate employee count: 49

PIPELINE READY FOR NEW-TASK EMPLOYEE RECOMMENDATION
